# EcoData · S2 · Fundamentos PySpark · Dimensión Aire (Lanzeloth)
Extracción con esquema explícito, transformaciones, agregaciones, RDD, evaluación perezosa y `explain(True)`. **Ejecutar primero** (genera `bronze/`).

In [1]:
import os, json, time, getpass, platform
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
spark=(SparkSession.builder.master("local[2]").appName("EcoData-U1-Aire")
       .config("spark.ui.enabled","false").config("spark.sql.session.timeZone","UTC").config("spark.sql.shuffle.partitions","4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Evidencia | usuario:",getpass.getuser(),"| equipo:",platform.node(),"| reloj:",time.strftime("%Y-%m-%d %H:%M:%S"))
print("Spark",spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 03:33:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Evidencia | usuario: root | equipo: d14c8d16aa21 | reloj: 2026-11-19 03:33:48
Spark 4.2.0


In [2]:
ORIGEN="aq_juliaca_2023.json"      # copia local de la respuesta de Open-Meteo (reproducible sin red)
# Alternativa en vivo (requiere red):
# https://air-quality-api.open-meteo.com/v1/air-quality?latitude=-15.5&longitude=-70.1&hourly=pm10,pm2_5&start_date=2023-01-01&end_date=2023-12-31
h=json.load(open(ORIGEN))["hourly"]
os.makedirs("bronze",exist_ok=True)
pd.DataFrame({"time":h["time"],"pm10":h["pm10"],"pm2_5":h["pm2_5"]}).to_csv("bronze/aire_juliaca_2023.csv",index=False)
esquema=StructType([StructField("time",StringType(),True),StructField("pm10",DoubleType(),True),StructField("pm2_5",DoubleType(),True)])
bronze=spark.read.csv("bronze/aire_juliaca_2023.csv",header=True,schema=esquema)
print("Bronze filas:",bronze.count()); bronze.printSchema(); bronze.show(5)

Bronze filas: 8760
root
 |-- time: string (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)

+----------------+----+-----+
|            time|pm10|pm2_5|
+----------------+----+-----+
|2023-01-01T00:00| 0.0|  0.0|
|2023-01-01T01:00| 0.3|  0.2|
|2023-01-01T02:00| 0.5|  0.4|
|2023-01-01T03:00| 0.4|  0.3|
|2023-01-01T04:00| 0.4|  0.3|
+----------------+----+-----+
only showing top 5 rows


In [3]:
t=(bronze.withColumn("ts",F.to_timestamp("time"))
         .withColumn("mes",F.date_format("ts","yyyy-MM"))
         .withColumn("hora",F.hour("ts"))
         .withColumn("nivel_riesgo",F.when(F.col("pm2_5")>=50,"Alto").when(F.col("pm2_5")>=25,"Moderado").otherwise("Bajo")))
# Evaluación perezosa: hasta aquí NO se ha ejecutado nada
t0=time.time(); plan=t.filter(F.col("pm2_5")>10); print("transformación definida en %.4f s (sin acción)"%(time.time()-t0))
t0=time.time(); n=plan.count(); print("acción count() = %d en %.2f s"%(n,time.time()-t0))

transformación definida en 0.0211 s (sin acción)
acción count() = 626 en 0.36 s


In [4]:
mensual=(t.groupBy("mes").agg(F.count("*").alias("horas"),F.round(F.avg("pm2_5"),2).alias("pm25_media"),
          F.round(F.expr("percentile_approx(pm2_5,0.95)"),2).alias("pm25_p95"),F.round(F.max("pm2_5"),1).alias("pm25_max"),
          F.round(F.avg("pm10"),2).alias("pm10_media")).orderBy("mes"))
mensual.show(12,truncate=False)
t.groupBy("nivel_riesgo").agg(F.count("*").alias("horas"),F.round(F.avg("pm2_5"),2).alias("pm25_media")).show()

+-------+-----+----------+--------+--------+----------+
|mes    |horas|pm25_media|pm25_p95|pm25_max|pm10_media|
+-------+-----+----------+--------+--------+----------+
|2023-01|744  |1.63      |3.5     |5.8     |2.4       |
|2023-02|672  |4.77      |10.3    |13.5    |6.84      |
|2023-03|744  |4.33      |8.6     |11.6    |6.26      |
|2023-04|720  |5.73      |9.5     |16.4    |8.31      |
|2023-05|744  |5.87      |11.3    |21.6    |8.57      |
|2023-06|720  |5.21      |9.3     |12.3    |7.61      |
|2023-07|744  |5.94      |12.3    |17.8    |8.7       |
|2023-08|744  |6.82      |12.9    |32.7    |10.03     |
|2023-09|720  |5.79      |11.3    |35.2    |8.46      |
|2023-10|744  |5.61      |11.6    |24.6    |8.16      |
|2023-11|720  |5.48      |11.4    |27.6    |8.08      |
|2023-12|744  |4.95      |11.7    |17.0    |8.13      |
+-------+-----+----------+--------+--------+----------+

+------------+-----+----------+
|nivel_riesgo|horas|pm25_media|
+------------+-----+----------+
|      

In [5]:
print("=== explain(True) de la agregación mensual ===")
mensual.explain(True)

=== explain(True) de la agregación mensual ===
== Parsed Logical Plan ==
'Sort ['mes ASC NULLS FIRST], true
+- Aggregate [mes#25], [mes#25, count(1) AS horas#40L, round(avg(pm2_5#2), 2) AS pm25_media#41, round(percentile_approx(pm2_5#2, cast(0.95 as double), 10000, 0, 0), 2) AS pm25_p95#42, round(max(pm2_5#2), 1) AS pm25_max#43, round(avg(pm10#1), 2) AS pm10_media#44]
   +- Project [time#0, pm10#1, pm2_5#2, ts#24, mes#25, hora#26, CASE WHEN (pm2_5#2 >= cast(50 as double)) THEN Alto WHEN (pm2_5#2 >= cast(25 as double)) THEN Moderado ELSE Bajo END AS nivel_riesgo#27]
      +- Project [time#0, pm10#1, pm2_5#2, ts#24, mes#25, hour(ts#24, Some(UTC)) AS hora#26]
         +- Project [time#0, pm10#1, pm2_5#2, ts#24, date_format(ts#24, yyyy-MM, Some(UTC)) AS mes#25]
            +- Project [time#0, pm10#1, pm2_5#2, to_timestamp(time#0, None, TimestampType, Some(UTC), true) AS ts#24]
               +- Relation [time#0,pm10#1,pm2_5#2] csv

== Analyzed Logical Plan ==
mes: string, horas: bigint, pm

In [6]:
# RDD: horas por nivel de riesgo con map/reduceByKey
rdd=t.select("nivel_riesgo").rdd.map(lambda r:(r[0],1)).reduceByKey(lambda a,b:a+b)
print(sorted(rdd.collect()))

[Stage 13:>                                                         (0 + 1) / 1]

[('Bajo', 8747), ('Moderado', 13)]
